# Amazon Product Funnel Analysis — Data Pipeline

End-to-end pipeline: **API extraction → cleaning → feature engineering → CSV export**

- **Data source:** Axesso Real-Time Amazon Data API (RapidAPI)
- **Pull date:** 19/04/2026
- **Categories:** board games · coffee beans · wireless headphones
- **Records:** 224 unique product listings

In [1]:
import requests
import pandas as pd
import numpy as np
import time

## 1. API Setup

Credentials and endpoint configuration for the Axesso Amazon Data API via RapidAPI.

In [ ]:
API_KEY = "YOUR_API_KEY"  # Replace with your RapidAPI key
HOST = "axesso-axesso-amazon-data-service-v1.p.rapidapi.com"
url = "https://axesso-axesso-amazon-data-service-v1.p.rapidapi.com/amz/amazon-search-by-keyword-asin"

headers = {
    "x-rapidapi-key": API_KEY,
    "x-rapidapi-host": HOST
}

## 2. Data Extraction

Pulls 2 pages per category across 3 categories (~224 products total).
Each product is tagged with its source category before appending to the master list.
A 1-second delay between requests respects the free-tier rate limit.

> **Note:**  
> The API extraction step is preserved for reproducibility and documentation purposes, but was not rerun during the final cleanup phase of this project.  
>
> Amazon product listings, pricing, and review counts change continuously over time, meaning a fresh API pull would produce a materially different dataset from the original April 2026 snapshot used throughout the SQL analysis and dashboarding stages.
>
> The cleaned datasets stored in `/data/` are therefore treated as the fixed analytical baseline for this project.

In [ ]:
categories = ["board game", "coffee beans", "wireless headphones"]
all_products = []

for category in categories:
    print(f"\nFetching: {category}")
    for page in range(1, 3):
        print(f"  Page: {page}")
        params = {
            "keyword": category,
            "page": str(page),
            "domainCode": "com",
            "excludeSponsored": "false",
            "sortBy": "relevanceblender",
            "withCache": "true"
        }
        response = requests.get(url, headers=headers, params=params)
        print(f"  Status: {response.status_code}")
        products = response.json().get("searchProductDetails", [])
        print(f"  Products found: {len(products)}")
        for p in products:
            p["category"] = category
            all_products.append(p)
        time.sleep(1)

df = pd.DataFrame(all_products)
print(f"\nTotal records: {df.shape[0]} rows x {df.shape[1]} cols")
df["category"].value_counts()

### Raw Data Preservation

The original API pull was preserved as a raw CSV export immediately after extraction.

Because the Amazon API data changes continuously over time (prices, reviews, rankings, availability), the pipeline notebook is documented for reproducibility, but the extraction step is not intended to be re-run for this portfolio snapshot.

All subsequent cleaning and feature engineering steps operate on derived copies of the raw dataset, preserving the original source file unchanged.

In [ ]:
# Preserve untouched raw extraction snapshot
df.to_csv("raw_amazon_products.csv", index=False)

print("Raw dataset preserved:")
print("raw_amazon_products.csv")

### Reload Raw Snapshot

To ensure reproducibility without re-calling the API, the remainder of the notebook reloads the preserved raw CSV snapshot.

This allows all downstream cleaning and feature engineering steps to be rerun independently from the external API.

In [2]:
# Reload preserved raw dataset
df = pd.read_csv(r"C:\Users\user\Desktop\Data_Analytics_Project2\Data files(CSVs)\Day 1\raw_amazon_products.csv")

print("Raw snapshot reloaded successfully.")
print(df.shape)

df.head()

Raw snapshot reloaded successfully.
(224, 17)


,productDescription,asin,countReview,imgUrl,price,retailPrice,productRating,prime,dpUrl,series,deliveryMessage,variations,productDetails,salesVolume,manufacturer,secondaryOffer,category
0,"CATAN Board Game (6th Edition) Trade, Build & ...",B0DYK1ZH2D,39443,https://m.media-amazon.com/images/I/71AbDpYEkg...,41.99,43.99,4.8 out of 5 stars,False,/CATAN-Classic-Strategy-Players-Playtime/dp/B0...,NaN,"FREE delivery Sat, Apr 25",[],[],6K+ bought in past month,NaN,0.00,board game
1,"Hasbro Gaming Connect 4 Classic Grid, 4 in a R...",B00D8STBHY,81275,https://m.media-amazon.com/images/I/81fEiLrmZ8...,8.89,9.99,4.8 out of 5 stars,False,/Hasbro-A5640-Connect-4-Game/dp/B00D8STBHY/ref...,NaN,"FREE delivery Sat, Apr 25 on $35 of items ship...",[],[],10K+ bought in past month,NaN,0.00,board game
2,Sorry! Board Game for Kids Ages 6 and Up; Clas...,B076HK9H7Z,33742,https://m.media-amazon.com/images/I/81CA3GV9sX...,8.98,9.99,4.8 out of 5 stars,False,/Hasbro-Gaming-A5065-Sorry-Game/dp/B076HK9H7Z/...,NaN,"FREE delivery Sat, Apr 25 on $35 of items ship...",[],[],10K+ bought in past month,NaN,0.00,board game
3,HUES and CUES - Vibrant Color Guessing Board G...,B084D2XQ9F,9848,https://m.media-amazon.com/images/I/81WUD2cZo5...,24.95,0.00,4.7 out of 5 stars,False,/USAOPOLY-Vibrant-Guessing-Perfect-Together/dp...,NaN,"FREE delivery Sat, Apr 25 on $35 of items ship...",[],[],8K+ bought in past month,NaN,22.07,board game
4,Hasbro Gaming Candy Land Kingdom of Sweet Adve...,B00000DMF5,37195,https://m.media-amazon.com/images/I/71ZnGyItfo...,12.99,0.00,4.8 out of 5 stars,False,/Hasbro-Gaming-Kingdom-Adventures-Exclusive/dp...,NaN,"FREE delivery Sat, Apr 25 on $35 of items ship...",[],[],10K+ bought in past month,NaN,0.00,board game


## 3. Data Cleaning

All transformations are applied to a working copy. The raw file remains untouched.

### 3.1 Null Audit

In [3]:
def data_check(df):
    print("SHAPE:", df.shape)
    print("\nNULLS (top 10):")
    print(df.isnull().sum().sort_values(ascending=False).head(10))
    print("\nDTYPES:")
    print(df.dtypes)

data_check(df)

SHAPE: (224, 17)

NULLS (top 10):
series                224
manufacturer          224
salesVolume             5
deliveryMessage         5
productRating           1
productDescription      0
secondaryOffer          0
productDetails          0
variations              0
dpUrl                   0
dtype: int64

DTYPES:
productDescription     object
asin                   object
countReview             int64
imgUrl                 object
price                 float64
retailPrice           float64
productRating          object
prime                    bool
dpUrl                  object
series                float64
deliveryMessage        object
variations             object
productDetails         object
salesVolume            object
manufacturer          float64
secondaryOffer        float64
category               object
dtype: object


### 3.2 Price — Replace Zeros, Cast to Float

The API returns `0.0` when price is unavailable. These are replaced with `NaN`.

In [4]:
df["price"] = df["price"].replace(0, np.nan)
df["price"] = pd.to_numeric(df["price"], errors="coerce")

print(f"Price nulls after cleaning: {df['price'].isnull().sum()}")
df["price"].describe()

Price nulls after cleaning: 6


count    218.000000
mean      35.632706
std       42.713926
min        4.990000
25%       16.990000
50%       24.335000
75%       37.987500
max      359.000000
Name: price, dtype: float64

### 3.3 Product Rating — Extract Numeric Value

Raw format: `"4.8 out of 5 stars"` → parsed to `4.8` (float64).

In [5]:
df["productRating"] = (
    df["productRating"]
    .str.extract(r"([0-9.]+)")
    .astype(float)
)
df["productRating"] = pd.to_numeric(df["productRating"], errors="coerce")

print(f"Rating nulls: {df['productRating'].isnull().sum()}")
df["productRating"].describe()

Rating nulls: 1


count    223.000000
mean       4.605381
std        0.176216
min        4.000000
25%        4.500000
50%        4.600000
75%        4.700000
max        5.000000
Name: productRating, dtype: float64

### 3.4 Sales Volume — Parse Bucketed Strings to Numeric

The API returns bucketed strings (e.g. `"10K+ bought in past month"`).
These are parsed to approximate numeric values (`10000`).
Values are directional, not exact — comparisons should be treated as such.

In [6]:
def clean_sales_volume(x):
    if pd.isna(x):
        return np.nan
    x = str(x).upper()
    num = ""
    for char in x:
        if char.isdigit():
            num += char
        elif char == ".":
            num += char
        elif num:
            break
    if not num:
        return np.nan
    value = float(num)
    if "K" in x:
        value *= 1000
    return value

df["salesVolume_clean"] = df["salesVolume"].apply(clean_sales_volume)

print(f"salesVolume_clean nulls: {df['salesVolume_clean'].isnull().sum()}")
df["salesVolume_clean"].describe()

salesVolume_clean nulls: 6


count      218.000000
mean      4532.798165
std       7559.922827
min         50.000000
25%       1000.000000
50%       2000.000000
75%       6000.000000
max      80000.000000
Name: salesVolume_clean, dtype: float64

## 4. Feature Engineering

### 4.1 Price Tier — Per-Category Quantile Segmentation

Tiers are defined **within each category** using tertile splits (`pd.qcut`).
This ensures a 'budget' wireless headphone is cheap relative to *other headphones*,
not relative to coffee beans — preserving category-specific pricing context.

In [7]:
df["category"] = df["category"].str.strip().str.lower()

df["price_tier"] = df.groupby("category")["price"].transform(
    lambda x: pd.qcut(x, q=3, labels=["budget", "mid", "premium"], duplicates="drop")
)

df["price_tier"].value_counts()

price_tier
budget     75
mid        72
premium    71
Name: count, dtype: int64

### 4.2 Engagement Features

In [8]:
# Binary flag: product has at least one review
df["has_reviews"] = (df["countReview"] > 0).astype(int)

# Review density: review count normalised by rating (small epsilon avoids division by zero)
df["review_density"] = df["countReview"] / (df["productRating"] + 0.01)

df[["has_reviews", "review_density"]].describe()

,has_reviews,review_density
count,224.000000,223.000000
mean,0.995536,2245.349157
std,0.066815,3940.404366
min,0.000000,3.821656
25%,1.000000,177.660218
50%,1.000000,701.587302
75%,1.000000,2572.434248
max,1.000000,26120.185615


## 5. Final Dataset — Column Selection & Export

In [9]:
df_final = df[[
    "asin",
    "productDescription",
    "category",
    "price",
    "retailPrice",
    "productRating",
    "countReview",
    "salesVolume_clean",
    "price_tier",
    "has_reviews",
    "review_density",
    "prime"
]].copy()

df_final.info()
df_final.sample(5, random_state=42)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 224 entries, 0 to 223
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   asin                224 non-null    object  
 1   productDescription  224 non-null    object  
 2   category            224 non-null    object  
 3   price               218 non-null    float64 
 4   retailPrice         224 non-null    float64 
 5   productRating       223 non-null    float64 
 6   countReview         224 non-null    int64   
 7   salesVolume_clean   218 non-null    float64 
 8   price_tier          218 non-null    category
 9   has_reviews         224 non-null    int64   
 10  review_density      223 non-null    float64 
 11  prime               224 non-null    bool    
dtypes: bool(1), category(1), float64(5), int64(2), object(3)
memory usage: 18.2+ KB


,asin,productDescription,category,price,retailPrice,productRating,countReview,salesVolume_clean,price_tier,has_reviews,review_density,prime
9,B09D4QRJ8Y,Hasbro Gaming Battleship Classic Board Game | ...,board game,16.84,19.99,4.7,7624,5000.0,budget,1,1618.683652,False
84,B00000DMFD,Mouse Trap Board Game For Kids Ages 6 and Up (...,board game,24.99,0.00,4.6,9324,1000.0,mid,1,2022.559653,False
117,B07LFSNK9M,"Starbucks Whole Coffee Beans, Medium Roast Hot...",coffee beans,30.42,32.07,4.7,1837,800.0,premium,1,390.021231,False
144,B074XXNDRC,"365 by Whole Foods Market, Pleasant Morning Bu...",coffee beans,18.48,0.00,4.2,1838,2000.0,budget,1,436.579572,False
221,B08D6GHYD7,Avantree Opera - Wireless Headphones for TV Wa...,wireless headphones,149.99,159.99,4.2,5268,NaN,premium,1,1251.306413,False


In [10]:
df_final.info()
df_final.sample(5, random_state=42)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 224 entries, 0 to 223
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   asin                224 non-null    object  
 1   productDescription  224 non-null    object  
 2   category            224 non-null    object  
 3   price               218 non-null    float64 
 4   retailPrice         224 non-null    float64 
 5   productRating       223 non-null    float64 
 6   countReview         224 non-null    int64   
 7   salesVolume_clean   218 non-null    float64 
 8   price_tier          218 non-null    category
 9   has_reviews         224 non-null    int64   
 10  review_density      223 non-null    float64 
 11  prime               224 non-null    bool    
dtypes: bool(1), category(1), float64(5), int64(2), object(3)
memory usage: 18.2+ KB


,asin,productDescription,category,price,retailPrice,productRating,countReview,salesVolume_clean,price_tier,has_reviews,review_density,prime
9,B09D4QRJ8Y,Hasbro Gaming Battleship Classic Board Game | ...,board game,16.84,19.99,4.7,7624,5000.0,budget,1,1618.683652,False
84,B00000DMFD,Mouse Trap Board Game For Kids Ages 6 and Up (...,board game,24.99,0.00,4.6,9324,1000.0,mid,1,2022.559653,False
117,B07LFSNK9M,"Starbucks Whole Coffee Beans, Medium Roast Hot...",coffee beans,30.42,32.07,4.7,1837,800.0,premium,1,390.021231,False
144,B074XXNDRC,"365 by Whole Foods Market, Pleasant Morning Bu...",coffee beans,18.48,0.00,4.2,1838,2000.0,budget,1,436.579572,False
221,B08D6GHYD7,Avantree Opera - Wireless Headphones for TV Wa...,wireless headphones,149.99,159.99,4.2,5268,NaN,premium,1,1251.306413,False


In [11]:
df_final.to_csv("cleanest_amazon_products.csv", index=False)
print("Cleaned dataset exported: cleanest_amazon_products.csv")

Cleaned dataset exported: cleanest_amazon_products.csv
